In [7]:
from pymongo import MongoClient, UpdateOne
import numpy as np
from tqdm.auto import tqdm
from sklearn.cluster import MiniBatchKMeans

import numpy as np
import pandas as pd

from pymongo import MongoClient, UpdateOne
from tqdm.auto import tqdm

from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score



In [8]:
host = "10.255.68.40"
port = 27017
db_name = "ejoow"

client = MongoClient(host=host, port=port)
db = client[db_name]

checkin_col = db["2. US_checkin"]
freq_col = db["3. user_cluster_frequency"]

print("checkins:", checkin_col.estimated_document_count())


checkins: 897701


1) (중요) 위치 클러스터 모델 준비

In [9]:
# ---------- 설정 ----------
LOC_K = 50          # 🔥 추천: 30~100 사이
BATCH = 20000

# ---------- 모델 ----------
loc_kmeans = MiniBatchKMeans(
    n_clusters=LOC_K,
    random_state=42,
    batch_size=BATCH,
    n_init="auto",
    max_iter=100,
    reassignment_ratio=0.01,
)
# ---------- partial_fit ----------
cursor = checkin_col.find(
    {"latitude": {"$ne": None}, "longitude": {"$ne": None}},
    {"latitude": 1, "longitude": 1},
    no_cursor_timeout=True
)

buf = []
total = checkin_col.count_documents(
    {"latitude": {"$ne": None}, "longitude": {"$ne": None}}
)

pbar = tqdm(total=total, desc="Training loc_kmeans")

for doc in cursor:
    buf.append([doc["latitude"], doc["longitude"]])

    if len(buf) >= BATCH:
        X = np.asarray(buf, dtype=np.float32)
        loc_kmeans.partial_fit(X)
        pbar.update(len(buf))
        buf.clear()

# flush
if buf:
    X = np.asarray(buf, dtype=np.float32)
    loc_kmeans.partial_fit(X)
    pbar.update(len(buf))

pbar.close()
cursor.close()

print("✅ loc_kmeans trained, k =", loc_kmeans.n_clusters)


/home/gpuadmin/.local/lib/python3.10/site-packages/pymongo/synchronous/collection.py:1945: UserWarning: use an explicit session with no_cursor_timeout=True otherwise the cursor may still timeout after 30 minutes, for more info see https://mongodb.com/docs/v4.4/reference/method/cursor.noCursorTimeout/#session-idle-timeout-overrides-nocursortimeout
  return Cursor(self, *args, **kwargs)


Training loc_kmeans:   0%|          | 0/897701 [00:00<?, ?it/s]

✅ loc_kmeans trained, k = 50


In [10]:
# 예시: 이미 만들어진/로드된 위치 클러스터 모델이라고 가정
# loc_kmeans = ...  # MiniBatchKMeans 모델

k = loc_kmeans.n_clusters
print("location k =", k)


location k = 50


2) 체크인 문서에 loc_cluster_id 붙이기 (진행률 포함)

In [11]:
from pymongo import UpdateOne

BATCH = 20000  # 상황에 맞게 5천~5만 사이 조절

cursor = checkin_col.find(
    {"latitude": {"$ne": None}, "longitude": {"$ne": None}},
    {"_id": 1, "latitude": 1, "longitude": 1},
    no_cursor_timeout=True
)

ops = []
buf_xy = []
buf_ids = []

total = checkin_col.count_documents({"latitude": {"$ne": None}, "longitude": {"$ne": None}})
pbar = tqdm(total=total, desc="Assigning loc_cluster_id")

for doc in cursor:
    buf_ids.append(doc["_id"])
    buf_xy.append([doc["latitude"], doc["longitude"]])

    if len(buf_ids) >= BATCH:
        X = np.asarray(buf_xy, dtype=np.float32)
        labels = loc_kmeans.predict(X)

        for _id, c in zip(buf_ids, labels):
            ops.append(UpdateOne({"_id": _id}, {"$set": {"loc_cluster_id": int(c)}}))

        checkin_col.bulk_write(ops, ordered=False)

        pbar.update(len(buf_ids))
        ops.clear()
        buf_xy.clear()
        buf_ids.clear()

# flush
if buf_ids:
    X = np.asarray(buf_xy, dtype=np.float32)
    labels = loc_kmeans.predict(X)
    for _id, c in zip(buf_ids, labels):
        ops.append(UpdateOne({"_id": _id}, {"$set": {"loc_cluster_id": int(c)}}))
    checkin_col.bulk_write(ops, ordered=False)
    pbar.update(len(buf_ids))

pbar.close()
cursor.close()
print("done: loc_cluster_id updated")


/home/gpuadmin/.local/lib/python3.10/site-packages/pymongo/synchronous/collection.py:1945: UserWarning: use an explicit session with no_cursor_timeout=True otherwise the cursor may still timeout after 30 minutes, for more info see https://mongodb.com/docs/v4.4/reference/method/cursor.noCursorTimeout/#session-idle-timeout-overrides-nocursortimeout
  return Cursor(self, *args, **kwargs)


Assigning loc_cluster_id:   0%|          | 0/897701 [00:00<?, ?it/s]

done: loc_cluster_id updated


3) user_id별 클러스터 방문 빈도 집계 → "3. user_cluster_frequency" 저장

In [13]:
freq_col = db["3. user_cluster_frequency"]

# 혹시 이전에 중복 user_id가 들어가 있으면 unique 인덱스 생성이 실패할 수 있어요.
# 그 경우 아래 "해결 B"를 쓰거나 컬렉션을 비우고 진행하세요.
freq_col.create_index("user_id", unique=True)
print("✅ unique index created on user_id")


pipeline = [
    # loc_cluster_id 있는 것만
    {"$match": {"loc_cluster_id": {"$exists": True}}},
    
    # (user_id, loc_cluster_id) 별 count
    {"$group": {
        "_id": {"user_id": "$user_id", "cluster_id": "$loc_cluster_id"},
        "count": {"$sum": 1}
    }},
    
    # user_id 별로 묶어서 배열 형태로 만들기
    {"$group": {
        "_id": "$_id.user_id",
        "total_checkins": {"$sum": "$count"},
        "clusters": {"$push": {"k": {"$toString": "$_id.cluster_id"}, "v": "$count"}}
    }},
    
    # clusters: [{"k":"0","v":12}, ...] -> frequency: {"0":12, ...}
    {"$addFields": {
        "frequency": {"$arrayToObject": "$clusters"},
        "k": k
    }},
    
    {"$project": {"_id": 0, "user_id": "$_id", "k": 1, "total_checkins": 1, "frequency": 1}},
    
    # 결과 저장
    {"$merge": {
        "into": "3. user_cluster_frequency",
        "on": "user_id",
        "whenMatched": "replace",
        "whenNotMatched": "insert"
    }}
]

# allowDiskUse 권장
list(checkin_col.aggregate(pipeline, allowDiskUse=True))
print("done: saved to 3. user_cluster_frequency")


✅ unique index created on user_id
done: saved to 3. user_cluster_frequency


In [14]:
from datetime import datetime, timezone

k = loc_kmeans.n_clusters  # 이미 학습한 위치 kmeans 기준
now = datetime.now(timezone.utc)

pipeline_freq_only = [
    {"$match": {"loc_cluster_id": {"$exists": True}}},

    # (user_id, loc_cluster_id) 별 체크인 수
    {"$group": {
        "_id": {"user_id": "$user_id", "cluster_id": "$loc_cluster_id"},
        "count": {"$sum": 1}
    }},

    # user_id별로 모아서 dict 만들기
    {"$group": {
        "_id": "$_id.user_id",
        "pairs": {"$push": {"k": {"$toString": "$_id.cluster_id"}, "v": "$count"}}
    }},

    {"$addFields": {
        "freq": {"$arrayToObject": "$pairs"},
        "user_id": "$_id",
        "k": k,
        "updated_at": now
    }},

    {"$project": {"pairs": 0}},  # 필요없는 중간필드 제거

    # ✅ _id=user_id 로 merge (유니크 보장)
    {"$merge": {
        "into": "3. user_cluster_frequency",
        "on": "_id",
        "whenMatched": "replace",
        "whenNotMatched": "insert"
    }}
]

list(checkin_col.aggregate(pipeline_freq_only, allowDiskUse=True))
print("✅ saved: 3. user_cluster_frequency")


✅ saved: 3. user_cluster_frequency
